# Study 764 -- SOPR 🔗
## For the quants: momentum regression, price horse race, per-band returns, >1/<1 timing vs buy-and-hold, time-shuffle placebo

*Part of [Open-Alpha-Lab](../../../README.md). See the [desk methodology](../../../METHODOLOGY.md).*

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Single-survivor: Named](https://img.shields.io/badge/Single--survivor-Named-8b949e?style=flat-square)

The headline numbers are frozen in `R` (mirror of [docs/results.md](../docs/results.md),
as-of 2026-07-13); the cells recompute them live from the cached BTC tape when present.


## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"

from sopr import data, strategy as st

CACHE_PATH = data.BTC_CACHE
HAVE_REAL = os.path.exists(CACHE_PATH)

if HAVE_REAL:
    df = data.joined_real(fetch=False, cache_path=CACHE_PATH)
    reg = st.predictive_regression(df)
    pos = st.timing_signal(df, thresh=1.0)
    bt = st.backtest_timing(df, pos, cost_bps=30.0)
    print(f"Real tape: {len(df)} aligned months  {df.index[0].date()} -> {df.index[-1].date()}")
else:
    df = reg = pos = bt = None
    print("No real BTC cache -- frozen headline numbers from R dict will be used")


Real tape: 142 aligned months  2014-09-30 -> 2026-06-30


In [2]:

# Frozen headline numbers (mirror of docs/results.md, as-of 2026-07-13)
R = {'n_months': 141, 'n_sopr': 150, 'aligned': 142, 'reg_slope': 0.9528, 'reg_t': 1.32, 'reg_r2': 0.013, 'horse_sopr_t': 0.18, 'horse_price_t': 1.04, 'band_greed_mo': 8.87, 'band_greed_n': 44, 'band_greed_hit': 0.59, 'band_neutral_mo': 4.75, 'band_neutral_n': 67, 'band_cap_mo': 2.6, 'band_cap_n': 30, 'band_cap_hit': 0.53, 'tim_share': 0.62, 'tim_turnover': 0.255, 'gross_ann': 60.6, 'gross_t': 2.66, 'timing_ann': 59.7, 'timing_t': 2.61, 'timing_sr': 0.98, 'bh_ann': 67.0, 'bh_t': 2.66, 'bh_sr': 0.93, 'excess_ann': -7.3, 'excess_t': -0.73, 'th99_ann': 59.5, 'th100_ann': 59.7, 'th101_ann': 54.9, 'placebo_real_mo': -0.61, 'placebo_mean_mo': -2.25, 'placebo_std_mo': 0.83, 'placebo_p': 0.975, 'syn_slope': 1.286, 'syn_t': 5.75, 'syn_null_t': -0.21}


## Positive control: the engine detects a planted momentum SOPR->price link

> 💡 **In plain words:** before trusting the harness on the real tape, we feed it
> fake data where we *know* the answer. If we plant "high SOPR this month -> higher
> return next month," the regression must find it; if we plant nothing, it must
> read ~zero. It does both. So a null on the real tape means *no signal*, not a
> broken tool.


In [3]:
# Positive control: beta=2.0 plants last-month SOPR stretch POSITIVELY into this-
# month price return (the momentum mechanism). SOPR's real stretch is tiny (~0.02
# in log), so a strong planted effect needs a large beta. Regression should
# recover a clearly positive slope.
df_syn, truth = data.synthetic_series(beta=2.0, seed=764)
reg_syn = st.predictive_regression(df_syn)
print(f"Positive control (beta=2.0): slope = {reg_syn['slope_sopr']:+.3f}  HAC t = {reg_syn['t_sopr']:+.2f}  n = {reg_syn['n']}")

# Null control: beta=0.0 -> SOPR is an independent mean-reverting series
df_null, _ = data.synthetic_series(beta=0.0, seed=764)
reg_null = st.predictive_regression(df_null)
print(f"Null control (beta=0.00):    slope = {reg_null['slope_sopr']:+.3f}  HAC t = {reg_null['t_sopr']:+.2f}")
print("\n-> Engine reads strongly positive on a planted momentum link, ~zero on null. It is truthful.")


Positive control (beta=2.0): slope = +1.286  HAC t = +5.75  n = 143
Null control (beta=0.00):    slope = -0.047  HAC t = -0.21

-> Engine reads strongly positive on a planted momentum link, ~zero on null. It is truthful.


## Real tape: predictive regression of next-month return on SOPR stretch

In [4]:
if HAVE_REAL:
    r = st.predictive_regression(df)
    print("r(t+1) = a + b*SOPR_stretch(t)")
    print(f"  slope={r['slope_sopr']:+.4f}  HAC t={r['t_sopr']:+.2f}  R^2={r['r2']:.4f}  n={r['n']}")
else:
    print(f"  slope={R['reg_slope']:+.4f}  HAC t={R['reg_t']:+.2f}  R^2={R['reg_r2']:.4f}  n={R['n_months']}")
print("\nThe slope is the right (positive) sign for the momentum story but does")
print("NOT clear |t| >= 2. SOPR stretch is not a robust leading indicator.")


r(t+1) = a + b*SOPR_stretch(t)
  slope=+0.9528  HAC t=+1.32  R^2=0.0128  n=141

The slope is the right (positive) sign for the momentum story but does
NOT clear |t| >= 2. SOPR stretch is not a robust leading indicator.


## Horse race: does SOPR add anything beyond BTC's own momentum?

> 💡 **In plain words:** SOPR is built from recent price action, so it might just
> be echoing "price went up." We put BTC's own one-month momentum in the same
> regression -- if SOPR's *t* collapses, it was never adding information.


In [5]:
if HAVE_REAL:
    rc = st.predictive_regression(df, add_price_control=True)
    print("r(t+1) = a + b*SOPR_stretch(t) + c*price_momentum(t)")
    print(f"  SOPR slope b: HAC t = {rc['t_sopr']:+.2f}")
    print(f"  price-mom slope c: HAC t = {rc['t_price']:+.2f}")
else:
    print(f"  SOPR slope: HAC t = {R['horse_sopr_t']:+.2f}")
    print(f"  price-mom slope: HAC t = {R['horse_price_t']:+.2f}")
print("\nWith price momentum in the regression, SOPR's t-stat collapses toward 0.")
print("Its faint directional whiff is not incremental to the price trend.")


r(t+1) = a + b*SOPR_stretch(t) + c*price_momentum(t)
  SOPR slope b: HAC t = +0.18
  price-mom slope c: HAC t = +1.04

With price momentum in the regression, SOPR's t-stat collapses toward 0.
Its faint directional whiff is not incremental to the price trend.


## Per-band forward returns: monotone but insignificant

In [6]:
if HAVE_REAL:
    tab = st.state_forward_stats(df, high=1.02, low=0.98)
    print(tab.to_string())
else:
    print("greed        : %+.2f%%/mo  hit=%.2f  n=%d" % (R['band_greed_mo'], R['band_greed_hit'], R['band_greed_n']))
    print("neutral      : %+.2f%%/mo            n=%d" % (R['band_neutral_mo'], R['band_neutral_n']))
    print("capitulation : %+.2f%%/mo  hit=%.2f  n=%d" % (R['band_cap_mo'], R['band_cap_hit'], R['band_cap_n']))
print("\nThe ordering is monotone in the folk direction (greed > neutral > capit.),")
print("but the gap is not significant (regression t=1.3) and dies in the horse race.")


                  mean   n       hit
band                                
greed         0.088673  44  0.590909
neutral       0.047541  67  0.537313
capitulation  0.026017  30  0.533333

The ordering is monotone in the folk direction (greed > neutral > capit.),
but the gap is not significant (regression t=1.3) and dies in the horse race.


## The '>1 / <1' regime timing rule vs buy-and-hold (net of costs)

In [7]:
if HAVE_REAL:
    s_net = st.summarize(bt["net"]); s_gross = st.summarize(bt["gross"]); s_bh = st.summarize(bt["bh"])
    print(f"Time in market: {st.time_in_market(pos):.1%}   avg turnover: {st.turnover(pos):.3f}/mo")
    print(f"GROSS timing: {s_gross['mean']*1200:+.1f}%/yr  SR={s_gross['sharpe']*12**0.5:+.2f}  HAC t={s_gross['tstat']:+.2f}")
    print(f"NET   timing: {s_net['mean']*1200:+.1f}%/yr  SR={s_net['sharpe']*12**0.5:+.2f}  HAC t={s_net['tstat']:+.2f}")
    print(f"BUY-HOLD:     {s_bh['mean']*1200:+.1f}%/yr  SR={s_bh['sharpe']*12**0.5:+.2f}  HAC t={s_bh['tstat']:+.2f}")
    excess = (bt['net'] - bt['bh'])
    se = st.summarize(excess)
    print(f"\nTiming minus buy-hold: {se['mean']*1200:+.1f}%/yr  HAC t={se['tstat']:+.2f}")
else:
    print(f"Time in market: {R['tim_share']:.1%}   avg turnover: {R['tim_turnover']:.3f}/mo")
    print(f"GROSS timing: {R['gross_ann']:+.1f}%/yr  HAC t={R['gross_t']:+.2f}")
    print(f"NET   timing: {R['timing_ann']:+.1f}%/yr  SR={R['timing_sr']:+.2f}  HAC t={R['timing_t']:+.2f}")
    print(f"BUY-HOLD:     {R['bh_ann']:+.1f}%/yr  SR={R['bh_sr']:+.2f}  HAC t={R['bh_t']:+.2f}")
    print(f"\nTiming minus buy-hold: {R['excess_ann']:+.1f}%/yr  HAC t={R['excess_t']:+.2f}")
print("\nThe rule is out of the market 38% of the time and LOSES ~7%/yr to holding")
print("(HAC t ~ -0.7). It buys a slightly higher Sharpe by forfeiting return -- on a")
print("150x asset that is a bad trade, not an edge.")


Time in market: 62.0%   avg turnover: 0.255/mo
GROSS timing: +60.6%/yr  SR=+0.99  HAC t=+2.66
NET   timing: +59.7%/yr  SR=+0.98  HAC t=+2.61
BUY-HOLD:     +67.0%/yr  SR=+0.93  HAC t=+2.66

Timing minus buy-hold: -7.3%/yr  HAC t=-0.73

The rule is out of the market 38% of the time and LOSES ~7%/yr to holding
(HAC t ~ -0.7). It buys a slightly higher Sharpe by forfeiting return -- on a
150x asset that is a bad trade, not an edge.


## Threshold sensitivity: no winning knob

In [8]:
if HAVE_REAL:
    for th in (0.99, 1.00, 1.01):
        p = st.timing_signal(df, thresh=th)
        b = st.backtest_timing(df, p, cost_bps=30.0)
        s = st.summarize(b["net"])
        print(f"thresh={th:.2f}: net {s['mean']*1200:+.1f}%/yr  SR={s['sharpe']*12**0.5:+.2f}  long {st.time_in_market(p):.0%}")
    s_bh = st.summarize(bt["bh"])
    print(f"buy-hold  : {s_bh['mean']*1200:+.1f}%/yr  SR={s_bh['sharpe']*12**0.5:+.2f}")
else:
    print(f"thresh=0.99: net {R['th99_ann']:+.1f}%/yr")
    print(f"thresh=1.00: net {R['th100_ann']:+.1f}%/yr  (the folk value)")
    print(f"thresh=1.01: net {R['th101_ann']:+.1f}%/yr")
    print(f"buy-hold   : {R['bh_ann']:+.1f}%/yr")
print("\nEvery threshold trails buy-and-hold. There is no value of the knob at which")
print("the rule wins -- the signature of a mirage, not a robust edge.")


thresh=0.99: net +59.5%/yr  SR=+0.92  long 78%
thresh=1.00: net +59.7%/yr  SR=+0.98  long 62%
thresh=1.01: net +54.9%/yr  SR=+0.96  long 49%
buy-hold  : +67.0%/yr  SR=+0.93

Every threshold trails buy-and-hold. There is no value of the knob at which
the rule wins -- the signature of a mirage, not a robust edge.


## Placebo: shuffle SOPR in time

> 💡 **In plain words:** if we scramble the SOPR values so they no longer line up
> with the right months, how well does the rule do? A *random* long/flat schedule
> that is long ~62% of the time still loses to buy-and-hold, because missing part
> of a strong uptrend costs money. If the real rule's result sits inside that
> random cloud, SOPR added nothing.


In [9]:
if HAVE_REAL:
    pl = st.placebo_edge(df, n_shuffles=2000)
    real_mo, mean_mo, std_mo, pval = pl['real_edge_mo']*100, pl['placebo_mean_mo']*100, pl['placebo_std_mo']*100, pl['p_value']
else:
    real_mo, mean_mo, std_mo, pval = R['placebo_real_mo'], R['placebo_mean_mo'], R['placebo_std_mo'], R['placebo_p']
print(f"Real rule edge over buy-hold : {real_mo:+.2f}%/mo")
print(f"Placebo edge (shuffled SOPR) : {mean_mo:+.2f}%/mo  (std {std_mo:.2f}pp)")
print(f"Two-sided empirical p        : {pval:.3f}")
print("\nA random 62%-long schedule loses ~2.3%/mo to holding; the real SOPR rule loses")
print("LESS (~0.6%/mo), so SOPR does time better than a coin flip. But it STILL loses to")
print("buy-and-hold, and its edge is nowhere near the tail (p~0.98). 'Beats random,")
print("loses to holding' = a weak, untradable regime filter.")


Real rule edge over buy-hold : -0.61%/mo
Placebo edge (shuffled SOPR) : -2.25%/mo  (std 0.83pp)
Two-sided empirical p        : 0.975

A random 62%-long schedule loses ~2.3%/mo to holding; the real SOPR rule loses
LESS (~0.6%/mo), so SOPR does time better than a coin flip. But it STILL loses to
buy-and-hold, and its edge is nowhere near the tail (p~0.98). 'Beats random,
loses to holding' = a weak, untradable regime filter.


## Cost & lag honesty

In [10]:
print("Honesty checklist:")
print(" - Execution lag: SOPR known at month-end t, position held for month t+1 (1-month lag).")
print(" - Costs: 30 bps one-way charged on every flip (|delta position|) x NAV. Long/flat -> no borrow.")
print(" - Returns: PRICE-ONLY (BTC pays no yield); same basis for timing and buy-hold.")
print(" - Proxy label: the SOPR series is a DIGITISED PROXY of the public Glassnode aSOPR chart,")
print("   hardcoded in sopr/data.py -- NOT a live feed. Named as a proxy, never under a real-tape banner.")
print(" - Partial month: the join stops at 2026-06-30 (last SOPR month), dropping the in-progress July bar.")
print(" - Single-survivor: BTC is the one crypto that ~150x'd and SOPR is DERIVED from its own")
print("   on-chain spending. The regime thresholds are fitted to ~four cycle turns. NAMED on Signal axis.")


Honesty checklist:
 - Execution lag: SOPR known at month-end t, position held for month t+1 (1-month lag).
 - Costs: 30 bps one-way charged on every flip (|delta position|) x NAV. Long/flat -> no borrow.
 - Returns: PRICE-ONLY (BTC pays no yield); same basis for timing and buy-hold.
 - Proxy label: the SOPR series is a DIGITISED PROXY of the public Glassnode aSOPR chart,
   hardcoded in sopr/data.py -- NOT a live feed. Named as a proxy, never under a real-tape banner.
 - Partial month: the join stops at 2026-06-30 (last SOPR month), dropping the in-progress July bar.
 - Single-survivor: BTC is the one crypto that ~150x'd and SOPR is DERIVED from its own
   on-chain spending. The regime thresholds are fitted to ~four cycle turns. NAMED on Signal axis.


## Verdict

In [11]:
print("=== Study 764 -- SOPR ===")
print()
print("Signal: NONE")
print(f"  SOPR stretch does not robustly predict next-month BTC returns: HAC t = {R['reg_t']:+.2f}")
print(f"  (R^2 ~ {R['reg_r2']:.2f}); in a horse race vs price momentum the SOPR slope is t = {R['horse_sopr_t']:+.2f}.")
print(f"  The band ordering is monotone (greed {R['band_greed_mo']:+.1f} > capit. {R['band_cap_mo']:+.1f} %/mo) but no t clears 2.")
print()
print("Tradability: MIRAGE")
print(f"  The '>1 / <1' rule LOSES {R['excess_ann']:+.1f}%/yr to buy-and-hold (HAC t = {R['excess_t']:+.2f}) and")
print(f"  trails at every threshold. The placebo shows it beats a coin flip (p = {R['placebo_p']:.2f}) yet still")
print("  can't beat holding. Any CAGR is just long exposure to a 150x survivor, minus 38% time in cash.")
print()
print("Single-survivor: NAMED -- BTC is the surviving moonshot; SOPR is derived from its own spending.")
print()
print("Bottom line: None/Mirage -- a beloved on-chain gauge that is hindsight")
print("regime-labelling on a single survivor, with no incremental predictive content.")


=== Study 764 -- SOPR ===

Signal: NONE
  SOPR stretch does not robustly predict next-month BTC returns: HAC t = +1.32
  (R^2 ~ 0.01); in a horse race vs price momentum the SOPR slope is t = +0.18.
  The band ordering is monotone (greed +8.9 > capit. +2.6 %/mo) but no t clears 2.

Tradability: MIRAGE
  The '>1 / <1' rule LOSES -7.3%/yr to buy-and-hold (HAC t = -0.73) and
  trails at every threshold. The placebo shows it beats a coin flip (p = 0.97) yet still
  can't beat holding. Any CAGR is just long exposure to a 150x survivor, minus 38% time in cash.

Single-survivor: NAMED -- BTC is the surviving moonshot; SOPR is derived from its own spending.

Bottom line: None/Mirage -- a beloved on-chain gauge that is hindsight
regime-labelling on a single survivor, with no incremental predictive content.
